# In-Class Activity: Find and Fix Security Issues with Semgrep and Gemini 🔒

**Objective:** Scan a small web app for security issues with Semgrep, use Gemini to fix them, and check each fix with a re-scan and the app's tests.

You will:
1. Scan a FastAPI app with Semgrep.
2. Triage the findings and confirm some of them by exploiting them.
3. Check the app's dependencies for known vulnerabilities.
4. Use Gemini to fix issues, then verify each fix.
5. Write your own Semgrep rules for issues the default rules miss.

Let's get started! 🚀

## Part 1: Setup and Configuration

Install the tools and set up API access.
- [Semgrep](https://github.com/semgrep/semgrep): a static analysis tool. It searches code for patterns that match known bugs and security issues.
- [pip-audit](https://github.com/pypa/pip-audit): checks Python dependencies against a database of known vulnerabilities.

In [ ]:
%pip install -q semgrep pip-audit "google-genai>=1.50"
%pip install -q fastapi "sqlalchemy>=2" python-dotenv pytest httpx

Add your Gemini API key to Colab **Secrets** (🔑 in the left sidebar) as `GEMINI_API_KEY`. Get a key [here](https://ai.google.dev/gemini-api/docs/api-key) if you don't have one.

In [ ]:
from google.colab import userdata

API_KEY = userdata.get('GEMINI_API_KEY')
print(f"Using API Key: {API_KEY[:4]}...{API_KEY[-4:]}")  # Print only the first and last 4 characters for security

In [ ]:
!semgrep --version
!pip-audit --version
print("✅ Setup Complete!")

### ⚠️ A Note on Rate Limits

We're using `gemini-3.5-flash-lite` because it's a free-tier model, but rate limits still apply (and vary by model).

See the [Rate Limits docs](https://ai.google.dev/gemini-api/docs/rate-limits) for details. Three things to watch:

- **RPM (Requests Per Minute):** API calls allowed per 60-second window.
- **TPM (Tokens Per Minute):** input tokens allowed per 60-second window.
- **RPD (Requests Per Day):** total requests allowed per day (resets at midnight Pacific time).

Exceeding any one triggers a `429 RESOURCE_EXHAUSTED` error. If that happens, wait a few seconds and try again, and don't send too many requests in a short time.

---

## Part 2: Get the App

We'll scan a small notes app: a FastAPI backend with a SQLite database and a plain JavaScript frontend. It has security issues on purpose.

| Path | What it is |
|---|---|
| `backend/app/routers/notes.py` | Notes API, plus some `/debug/...` endpoints |
| `backend/app/routers/action_items.py` | Action items API |
| `backend/app/services/extract.py` | Helper that pulls action items out of text |
| `backend/app/main.py` | App setup: CORS, routes, static files |
| `frontend/app.js` | Frontend code that renders notes and action items |
| `requirements.txt` | Pinned dependency versions |

In [ ]:
!git clone -q --depth 1 --branch fall2025 https://github.com/mihail911/modern-software-dev-assignments.git
%cd modern-software-dev-assignments/week6
!ls

Run the tests before changing anything. Every fix you make later must keep them passing.

In [ ]:
!PYTHONPATH=. pytest -q backend/tests

---

## Part 3: Scan with Semgrep

Semgrep checks source code without running it. This is called static analysis, or SAST. Each rule describes a risky pattern, e.g. "`subprocess.run` called with `shell=True`".

We use four public rule packs from the [Semgrep Registry](https://semgrep.dev/explore):
- `p/default`: Semgrep's recommended security rules
- `p/python` and `p/javascript`: language-specific rules
- `p/secrets`: rules for hardcoded keys and passwords


In [ ]:
!semgrep scan --config p/default --config p/python --config p/javascript --config p/secrets --metrics off --quiet .

The text output is easy to read but hard to work with in code. The helper below runs the same scan and returns the results as JSON.

In [ ]:
import json
import os
import subprocess
import sys

RULESETS = ["p/default", "p/python", "p/javascript", "p/secrets"]


def run_semgrep(configs: list[str] = RULESETS, target: str = ".") -> list[dict]:
    """Run Semgrep and return the list of findings."""
    cmd = ["semgrep", "scan", "--metrics", "off", "--json", "--quiet", target]
    for config in configs:
        cmd += ["--config", config]
    out = subprocess.run(cmd, capture_output=True, text=True)
    return json.loads(out.stdout)["results"]


initial_findings = run_semgrep()
print(len(initial_findings), "findings")
print(json.dumps(initial_findings[0], indent=2)[:1500])

Each finding has a rule ID (`check_id`), a file (`path`), a line (`start.line`), and details under `extra` such as `severity` and `message`. Turn them into a table.

In [ ]:
import pandas as pd


def summarize(findings: list[dict]) -> pd.DataFrame:
    """Return one row per finding with columns: rule, file, line, severity, message.

    Hint: the short rule name is the last part of check_id, e.g. check_id.split(".")[-1].
    """
    rows = [
        {
            "rule": f["check_id"].split(".")[-1],
            "file": f["path"],
            "line": f["start"]["line"],
            "severity": f["extra"]["severity"],
            "message": f["extra"]["message"],
        }
        for f in findings
    ]
    return pd.DataFrame(rows, columns=["rule", "file", "line", "severity", "message"])


summarize(initial_findings)

---

## Part 4: Triage the Findings

Not every finding is a real problem. Triage means deciding, for each one, whether it's a real risk, how bad it is, and whether to fix it now.

A good way to check is to try the attack yourself. The helper below starts the app in a fresh process and sends one request.

In [ ]:
def probe(url: str, params: dict) -> None:
    """Send one GET request to a fresh copy of the app and print the response."""
    script = f"""
from fastapi.testclient import TestClient
from backend.app.main import app
with TestClient(app) as client:
    r = client.get({url!r}, params={params!r})
    print(r.status_code, r.text[:300])
"""
    out = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, env={**os.environ, "PYTHONPATH": "."})
    print(f"GET {url} {params}")
    print("  ->", out.stdout.strip() or out.stderr.strip()[-500:])


# Example: /debug/eval runs any Python expression the user sends.
probe("/notes/debug/eval", {"expr": "__import__('os').getcwd()"})

In [ ]:
# /notes/debug/run runs any shell command.
probe("/notes/debug/run", {"cmd": "whoami"})

# /notes/debug/read reads any file on the server.
probe("/notes/debug/read", {"path": "/etc/hostname"})

# SQL injection attempt on /notes/unsafe-search.
# This returns 422, not the injected rows: the route "/{note_id}" is registered before
# "/unsafe-search", so FastAPI matches "unsafe-search" as note_id and fails to parse it as an int.
# The request never reaches the vulnerable SQL.
probe("/notes/unsafe-search", {"q": "' OR 1=1 --"})

### 🤔 Questions (TODO)
1. `/notes/unsafe-search` returns 422 before the SQL runs. Is Semgrep's finding a false positive? Should you still fix it?
2. `/debug/read` lets anyone read any file, but Semgrep didn't report it. What does that tell you about relying on one tool?
3. Rank the findings from most to least urgent. What did you base the order on?

### Your Answers (Write your answer for each question)
1. It is not a false positive. The code really does build SQL from user input with an f-string, so the injection is real. It is only unreachable today by accident: `/{note_id}` is registered before `/unsafe-search`, so the request is matched as a note ID and rejected with 422 before the query runs. Reordering the routes or reusing that function would make it exploitable right away, so it should still be fixed (bound parameters) or the dead endpoint removed.
2. A clean scan does not mean the code is safe. Semgrep only reports what its rules describe, and none of the four packs has a rule for `open()` on a user-supplied path, even though the probe shows it is trivially exploitable. One tool is one layer: it needs to be combined with manual review, actually trying the attack, custom rules for our own code (Part 7), and dependency scanning.
3. Most to least urgent:
   1. `subprocess-shell-true` (`/debug/run`) and `eval-detected` (`/debug/eval`): remote code execution with no authentication, confirmed by the probes. (`/debug/read` belongs in this group too, even though Semgrep missed it.)
   2. `dynamic-urllib-use-detected` (`/debug/fetch`): reachable SSRF, and `file://` URLs make it another file read.
   3. `avoid-sqlalchemy-text` (`/unsafe-search`): SQL injection is severe, but it is currently unreachable because of the route order.
   4. `wildcard-cors`: any origin with credentials allowed is a bad configuration, but the app has no login or cookies yet, so there is little to steal.
   5. `insecure-hash-algorithm-md5`: MD5 is weak, but here it only hashes a debug query string, not passwords or anything security-relevant.

   I ordered them by impact if exploited (full server takeover > data access > misconfiguration), by how easy the exploit is (one unauthenticated GET request), and by whether the probe confirmed it is actually reachable. Semgrep's own severity label was only a starting point: `eval-detected` is labeled WARNING but is as bad as the ERROR findings.

---

## Part 5: Check the Dependencies

Your code can be safe and still ship a vulnerable library. Checking dependencies for known vulnerabilities is called software composition analysis, or SCA.


In [ ]:
!cat requirements.txt
!pip-audit -r requirements.txt --no-deps --disable-pip --progress-spinner off -f json -o audit.json

In [ ]:
from packaging.version import Version

audit = json.load(open("audit.json"))

# One row per vulnerable package:
#   package, version, num_vulns (count unique vulnerability IDs), lowest_fix (lowest version that fixes at least one)
rows = []
for dep in audit["dependencies"]:
    vulns = dep.get("vulns", [])
    if not vulns:
        continue
    fix_versions = [Version(v) for vuln in vulns for v in vuln["fix_versions"]]
    rows.append(
        {
            "package": dep["name"],
            "version": dep["version"],
            "num_vulns": len({vuln["id"] for vuln in vulns}),
            "lowest_fix": str(min(fix_versions)) if fix_versions else None,
        }
    )

audit_df = pd.DataFrame(rows, columns=["package", "version", "num_vulns", "lowest_fix"])
audit_df

The lowest fix version removes only the first vulnerability. To fix all of them, upgrade to a current release and re-run the tests.

---

## Part 6: Fix Issues with Gemini

Now use Gemini as your AI coding tool. For each finding:
1. Send Gemini the rule, the message, and the full file.
2. Get back a short explanation and the fixed file. We use [structured output](https://ai.google.dev/gemini-api/docs/structured-output) so the answer is always valid JSON.
3. Check the fix: re-run Semgrep on the file and run the tests.
4. Keep the fix only if the finding is gone and the tests pass. Otherwise, restore the original file.

Always review the diff yourself. Passing the checks doesn't prove the fix is good.

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel

gemini = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.5-flash-lite"


class Fix(BaseModel):
    explanation: str  # what the risk was and how the change fixes it
    fixed_code: str   # the full updated file


print("✅ Gemini client ready")

In [ ]:
def ask_gemini_for_fix(finding: dict) -> Fix:
    """Ask Gemini to fix one Semgrep finding and return a Fix.

    Hints:
        - Read the file at finding["path"].
        - Put the rule ID, the message, the line number, and the full file in the prompt.
        - Ask for the smallest change that fixes only this issue and keeps all other behavior the same.
        - config = types.GenerateContentConfig(response_mime_type="application/json", response_schema=Fix,
                                               automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
        - response = gemini.models.generate_content(model=MODEL, contents=prompt, config=config)
        - response.parsed is a Fix object.
    """
    path = finding["path"]
    with open(path) as f:
        source = f.read()

    prompt = f"""You are a security engineer fixing one finding from a Semgrep scan.

Rule ID: {finding["check_id"]}
Message: {finding["extra"]["message"]}
File: {path}
Line: {finding["start"]["line"]}

Make the smallest change that fixes only this issue. Fix the actual security risk, not just
the pattern the rule matches. Keep all other behavior, endpoints, and code exactly the same.

Return:
- explanation: 1-3 sentences on what the risk was and how your change fixes it.
- fixed_code: the full updated file, as plain source code with no Markdown code fences.

Full file contents:
----- BEGIN FILE -----
{source}
----- END FILE -----
"""

    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Fix,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    )
    response = gemini.models.generate_content(model=MODEL, contents=prompt, config=config)
    return response.parsed

In [ ]:
import difflib


def run_tests() -> bool:
    """Run the app's tests and return True if they pass."""
    out = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "backend/tests"],
        capture_output=True, text=True, env={**os.environ, "PYTHONPATH": "."},
    )
    lines = (out.stdout + out.stderr).strip().splitlines()
    print(lines[-1] if lines else "(no test output)")
    return out.returncode == 0


def apply_and_verify(finding: dict, configs: list[str] = RULESETS) -> bool:
    """Fix one finding with Gemini, check the fix, and keep it only if it passes.

    Steps:
        1. Save the original file contents.
        2. Call ask_gemini_for_fix(finding). Print the explanation and a diff
           (hint: difflib.unified_diff(original.splitlines(True), new.splitlines(True))).
        3. Write the fixed code to the file.
        4. Re-run run_semgrep(configs, target=path) and check that no finding has the same check_id.
        5. Run run_tests().
        6. If either check fails, write the original contents back.

    Returns:
        True if the fix was kept.
    """
    path = finding["path"]

    # 1. Save the original file contents.
    with open(path) as f:
        original = f.read()

    # 2. Ask Gemini for a fix, then show the explanation and the diff.
    fix = ask_gemini_for_fix(finding)
    new = fix.fixed_code
    print("Explanation:", fix.explanation)
    diff = difflib.unified_diff(
        original.splitlines(True), new.splitlines(True), fromfile=f"a/{path}", tofile=f"b/{path}"
    )
    print("".join(diff))

    # 3. Write the fixed code to the file.
    with open(path, "w") as f:
        f.write(new)

    # 4. Re-scan the file: the same rule must no longer match.
    finding_gone = all(f["check_id"] != finding["check_id"] for f in run_semgrep(configs, target=path))

    # 5. Run the tests.
    tests_pass = run_tests()

    # 6. Keep the fix only if both checks pass; otherwise restore the original.
    if finding_gone and tests_pass:
        print("✅ Fix kept")
        return True

    with open(path, "w") as f:
        f.write(original)
    print(f"❌ Fix reverted (finding gone: {finding_gone}, tests pass: {tests_pass})")
    return False

Pick **3 findings** to fix.

In [ ]:
# The three most urgent findings from the triage in Part 4.
rules_to_fix = ["subprocess-shell-true", "eval-detected", "avoid-sqlalchemy-text"]

for rule in rules_to_fix:
    finding = next((f for f in run_semgrep() if f["check_id"].endswith(rule)), None)
    if finding is None:
        print(f"{rule}: not found (already fixed?)")
        continue
    print(f"\n========== {rule} ({finding['path']}:{finding['start']['line']}) ==========")
    apply_and_verify(finding)

### 🤔 Questions (TODO)
1. Read each diff. Did Gemini fix the root cause, or did it just change the code so the rule stops matching?
2. Did Gemini delete any endpoint? Is that an acceptable fix?
3. What could a bad fix break that these tests wouldn't catch?

### Your Answers (Write your answer for each question)
1. It was mixed. The SQL fix is a root-cause fix: the query is now a constant string and the user input is passed as a bound parameter, so it can no longer change the SQL. The `eval` fix also removes the root cause, because `ast.literal_eval` only parses literals and cannot call functions or import modules. The `subprocess` fix is closer to "make the rule stop matching": dropping `shell=True` stops shell tricks like `;` and `|`, but the endpoint still runs whatever program the caller names, so remote command execution is still there. The Part 8 probe confirms it: `whoami` still works after the "fix".
2. No. The prompt asked for the smallest change that keeps all other behavior, so Gemini kept every endpoint. For these `/debug/...` endpoints, deleting them would actually be the best fix, since nothing in the frontend or the tests uses them and there is no safe version of "run any command". But it has to be a deliberate decision by a person. Deleting code always makes the finding disappear and keeps the tests green, so if the model did it silently to a real feature, our two checks would not notice.
3. A lot, because the tests only cover creating/listing/patching notes and action items and the `extract` helper. They never call `/unsafe-search` or any `/debug/...` endpoint, and they never load the frontend. A bad fix could return wrong search results (for example a broken `LIKE` pattern), change or remove an endpoint's response, turn bad input into a 500 error, or break `app.js` so notes do not render at all, and all three tests would still pass. The tests also say nothing about security: a fix can pass them and still be exploitable.

### Record Your Fixes

Fill in one row for each fix you kept (double-click to edit):

| Fix | File and lines | Semgrep rule | Risk | Your change | Why it fixes the risk |
|---|---|---|---|---|---|
| 1 | `backend/app/routers/notes.py`, `debug_run` (line 112) | `subprocess-shell-true` | Command injection: the `cmd` query parameter was run through a shell, so anyone could run any shell command on the server | Split the command with `shlex.split(cmd)` and call `subprocess.run` without `shell=True` | The input is no longer interpreted by a shell, so `;`, `\|`, and `$(...)` cannot chain extra commands. This is only a partial fix: the endpoint still runs any single program, so it should be removed before deploying |
| 2 | `backend/app/routers/notes.py`, `debug_eval` (line 104) | `eval-detected` | Remote code execution: `eval(expr)` ran any Python expression sent in the query string | Replaced `eval(expr)` with `ast.literal_eval(expr)` | `literal_eval` only accepts literals (numbers, strings, lists, dicts), so function calls and imports such as `__import__('os')` are rejected |
| 3 | `backend/app/routers/notes.py`, `unsafe_search` (lines 71-80) | `avoid-sqlalchemy-text` | SQL injection: `q` was inserted into the SQL string with an f-string | Removed the f-string, used a named placeholder in `text(...)`, and passed `q` through `db.execute(sql, {...})` as a bound parameter | The database treats the bound value as data only, so input like `' OR 1=1 --` can no longer change the structure of the query |

---

## Part 7: Write Your Own Semgrep Rules

The public rules missed some real issues, such as the file read in `/debug/read`. You can write rules for your own code in YAML. A rule's pattern looks like the code it matches:
- `$X` matches any expression and remembers it.
- `...` matches any sequence of arguments, statements, or characters.
- `metavariable-regex` adds a regex check on what a metavariable matched.

The first rule below is a complete example. Write the other two. You can test patterns in the [Semgrep Playground](https://semgrep.dev/playground) and read the [rule syntax docs](https://semgrep.dev/docs/writing-rules/rule-syntax).

In [ ]:
%%writefile custom_rules.yml
rules:
  # Example: flag any route whose path contains /debug/
  - id: fastapi-debug-endpoint
    languages: [python]
    severity: WARNING
    message: Debug endpoint $PATH is exposed. Remove it before deploying.
    patterns:
      - pattern: |
          @$ROUTER.$METHOD($PATH, ...)
          def $FUNC(...): ...
      - metavariable-regex:
          metavariable: $PATH
          regex: .*/debug/.*

  # Flag an assignment like NAME = "sk_live_..." (a string that starts with sk_live_).
  - id: hardcoded-live-api-key
    languages: [python]
    severity: ERROR
    message: Hardcoded live API key assigned to $NAME. Load it from an environment variable instead.
    patterns:
      - pattern: $NAME = "$VALUE"
      - metavariable-regex:
          metavariable: $VALUE
          regex: ^sk_live_.*

  # Flag assigning a template literal with ${...} to .innerHTML, e.g. li.innerHTML = `<b>${n.title}</b>`.
  # A plain string like list.innerHTML = '' has no ${...}, so it does not match.
  - id: innerhtml-template-literal
    languages: [javascript]
    severity: ERROR
    message: Template literal with interpolated data is assigned to innerHTML (XSS risk). Use textContent or build DOM nodes instead.
    pattern: $EL.innerHTML = `...${$X}...`

In [ ]:
custom_findings = run_semgrep(configs=["custom_rules.yml"])
summarize(custom_findings)

Your rules work with the same fix loop. Fix the XSS in `frontend/app.js` with Gemini.

In [ ]:
xss_finding = next(f for f in custom_findings if f["check_id"].endswith("innerhtml-template-literal"))
print(f"{xss_finding['path']}:{xss_finding['start']['line']}")

# Re-scan with the custom rules, since the public rules never reported this issue.
apply_and_verify(xss_finding, configs=["custom_rules.yml"])

---

## Part 8: Final Check

Re-scan with all the rules, then re-run the attacks from Part 4. Endpoints you fixed should now return an error or 404. Any that still work are not fixed yet.

In [ ]:
final_findings = run_semgrep(RULESETS + ["custom_rules.yml"])
print(f"Public rules: {len(initial_findings)} findings before, {len(run_semgrep())} after")
print(f"All rules now: {len(final_findings)} findings")
summarize(final_findings)

In [ ]:
probe("/notes/debug/eval", {"expr": "__import__('os').getcwd()"})
probe("/notes/debug/run", {"cmd": "whoami"})
probe("/notes/debug/read", {"path": "/etc/hostname"})

See every file you changed:

In [ ]:
!git diff --stat